# Underwater detection: Kaggle training
Enable GPU. Attach the extracted Figshare dataset and this repository source as inputs. Check the dataset license before any public redistribution. This notebook does not publish data. Reported 97.5% mAP is reference only.

In [ ]:
from pathlib import Path
import subprocess, sys, shutil, json
# EDIT to match your attached Kaggle inputs.
CODE = Path('/kaggle/input/drowning-repo/model_training/pipeline')
DATASET = Path('/kaggle/input/underwater-drowning/datasets')
GROUPS = None  # Optional CSV columns: image,group; image relative to images/.
PREPARED = Path('/kaggle/working/prepared')
OUTPUT = Path('/kaggle/working/experiments')
assert (CODE/'audit.py').exists(), 'Set CODE'
assert (DATASET/'images/train').exists(), 'Set DATASET'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(CODE/'requirements-kaggle.txt')], check=True)


Restart the Python session after installation if Torch/NumPy were already imported, then rerun path definitions. Internet is needed for dependencies/pretrained weights. Save a notebook version with outputs to retain checkpoints.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU accelerator'
import torchvision
print('Python:', sys.version)
print('Torch:', torch.__version__, 'TorchVision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
boxes = torch.tensor([[0., 0., 1., 1.]], device='cuda')
scores = torch.tensor([1.], device='cuda')
assert torchvision.ops.nms(boxes, scores, 0.5).numel() == 1
command=[sys.executable,str(CODE/'audit.py'),str(DATASET),'--output',str(PREPARED)]
if GROUPS: command += ['--groups',str(GROUPS)]
subprocess.run(command,check=True)


In [ ]:
from IPython.display import display, Image
report=json.loads((PREPARED/'audit.json').read_text())
print(json.dumps({k:v for k,v in report.items() if k not in ('near_cross_split','exact_cross_split')},indent=2))
print('Near-duplicate cross-split pairs:',len(report['near_cross_split']))
display(Image(filename=str(PREPARED/'annotations.jpg')))


Review annotation previews and audit.json before training. Exact cross-split duplicates/invalid labels stop preparation. With no source groups, retain original splits and report their limitations; do not present validation as independent test data.

In [ ]:
# Resume: attach a previous saved notebook output, then copy experiments/ to OUTPUT.
# shutil.copytree('/kaggle/input/previous-run/experiments',OUTPUT,dirs_exist_ok=True)
subprocess.run([sys.executable,str(CODE/'train.py'),'--data',str(PREPARED/'data.yaml'),
                '--output',str(OUTPUT),'--resume'],check=True)


In [ ]:
comparison=json.loads((OUTPUT/'comparison.json').read_text())
selected=json.loads((OUTPUT/'selected.json').read_text())
print(json.dumps(comparison,indent=2))
print('Selected by validation:',selected['checkpoint'])


## Held-out test
Run only after freezing selection. Skip if data.yaml has no independent test split. Test scores never choose weights.

In [ ]:
RUN_HELD_OUT_TEST=False
if RUN_HELD_OUT_TEST:
    subprocess.run([sys.executable,str(CODE/'train.py'),'--data',str(PREPARED/'data.yaml'),
                    '--output',str(OUTPUT),'--test-checkpoint',selected['checkpoint']],check=True)


## Deploy and rollback
Download experiments/ including underwater_selected.pt, selected.json, comparison.json, environment.json, audit/manifest, plots and checkpoints. Set UNDERWATER_MODEL to the new checkpoint; keep underwaterModel.pt untouched. Evaluate independent labeled videos with evaluate_events.py. Compare GAN off/on on identical inputs. No accuracy or soak result is claimed until an actual run completes.